In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
%matplotlib inline

In [ ]:
def load_dataset():
    with open("../data/train_data.json", "r") as f:
        train = json.load(f)
    with open("../data/val_data.json", "r") as f:
        val = json.load(f)
    with open("../data/test_data.json", "r") as f:
        test = json.load(f)
    
    train_df = pd.DataFrame(train["data"])
    val_df = pd.DataFrame(val["data"])
    test_df = pd.DataFrame(test["data"])
    
    return {'train': train_df, 'val': val_df, 'test': test_df}

In [ ]:
datasets = load_dataset()

In [ ]:
def plot_data_distribution(df, ax, xlabel=None, ylabel=None, category='my_label', kind='bar'):
    df.groupby(category).size().plot(
        kind=kind, color=sns.color_palette("Dark2"), ax=ax,
        xlabel=xlabel, ylabel=ylabel)
    ax.spines[['top', 'right']].set_visible(False)

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.capitalize()} dataset shape: {df.shape}")
    print(f"{name.capitalize()} dataset columns: {df.columns.tolist()}")
    if 'my_label' in df.columns:
        print(f"{name.capitalize()} label counts:")
        print(df['my_label'].value_counts())

plt.figure(figsize=(18, 5))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Label Distribution Across Datasets', fontsize=16)

for i, (name, df) in enumerate(datasets.items()):
    plot_data_distribution(
        df, axes[i], 
        xlabel='Label', 
        ylabel='Count' if i == 0 else None,
        category='my_label'
    )
    axes[i].set_title(f'{name.capitalize()} Set')

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

content_col = 'text'

plt.figure(figsize=(18, 10))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Content Length Analysis by Label', fontsize=16)

avg_lengths_df = pd.DataFrame()

for i, (name, df) in enumerate(datasets.items()):
    if content_col in df.columns:
        df['content_length'] = df[content_col].apply(lambda x: len(str(x).split()))
        
        avg_length = df.groupby('my_label')['content_length'].mean()
        print(f"\n{name.capitalize()} Set - Average Content Length per Label:")
        print(avg_length)
        
        avg_lengths_df[name] = avg_length
        
        avg_length.plot(
            kind='bar', 
            ax=axes[0, i], 
            color=sns.color_palette("Dark2"),
            xlabel='Label', 
            ylabel='Average Word Count' if i == 0 else None
        )
        axes[0, i].set_title(f'{name.capitalize()} Set - Average Content Length')
        axes[0, i].spines[['top', 'right']].set_visible(False)
        
        sns.boxplot(
            x='my_label', 
            y='content_length', 
            data=df, 
            ax=axes[1, i],
            palette="Dark2"
        )
        axes[1, i].set_title(f'{name.capitalize()} Set - Content Length Distribution')
        axes[1, i].set_xlabel('Label')
        axes[1, i].set_ylabel('Word Count' if i == 0 else None)
        axes[1, i].spines[['top', 'right']].set_visible(False)
    else:
        axes[0, i].text(0.5, 0.5, f'Text column not found in {name} dataset', 
                     ha='center', va='center', fontsize=12)
        axes[1, i].text(0.5, 0.5, f'Text column not found in {name} dataset', 
                     ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

if not avg_lengths_df.empty:
    print("\nAverage Content Length Comparison Across Datasets:")
    print(avg_lengths_df)

if all(content_col in df.columns for df in datasets.values()):
    plt.figure(figsize=(18, 15))
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle('Content Length Histograms by Label', fontsize=16)
    
    labels = ['neutral', 'moderate', 'high-risk']
    
    for i, (name, df) in enumerate(datasets.items()):
        for j, label in enumerate(labels):
            label_data = df[df['my_label'] == label]['content_length']
            if not label_data.empty:
                sns.histplot(
                    label_data,
                    ax=axes[j, i], 
                    kde=True,
                    color=sns.color_palette("Dark2")[j]
                )
                axes[j, i].set_title(f'{name.capitalize()} - {label}')
                axes[j, i].set_xlabel('Word Count' if j == 2 else '')
                axes[j, i].set_ylabel('Frequency' if i == 0 else '')
                axes[j, i].spines[['top', 'right']].set_visible(False)
            else:
                axes[j, i].text(0.5, 0.5, f'No {label} examples in {name} dataset', 
                            ha='center', va='center', fontsize=12)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

for name, df in datasets.items():
    if content_col in df.columns:
        print(f"\n{name.capitalize()} Set - Content Length Statistics by Label:")
        stats = df.groupby('my_label')['content_length'].describe()
        print(stats)